In [1]:
from google.colab import files
uploaded = files.upload()

Saving esophageal1_male.csv to esophageal1_male.csv
Saving esophageal2_female.csv to esophageal2_female.csv
Saving midyear1_pop.csv to midyear1_pop.csv


In [2]:
"""
================================================================

  資料集（2 組，皆為 age_level(1–18) × year(1979–2023) 的 SAS FREQ 交叉表）：
    - esophageal_male     食道癌．男    esophageal1_male.csv
    - esophageal_female   食道癌．女    esophageal2_male.csv

  人口分母：midyear1_pop.csv，依癌別的性別篩選 SEX（1=男 / 2=女），
  TPOP1–TPOP18 對應相同 18 個年齡組。

★ 設定與計算方式★
  - WHO 2000 世界標準人口權重（附錄四）
  - ASIR_t = Σ WHO_W_i (d_it/n_it) × 100,000；Delta method 算 Var(log ASIR_t) 當共用加權
  - 六方法：Linear / Quadratic / Trad-RCS / Joinpoint（NCI ka–kb inward）/ DL-ReLU / DL-RCS
  - DL 超參數、bowl penalty、WLS 暖啟動、RCS 節點建構，皆與模擬情境 thesis_v4_final.py 相同
  - y 軸 log scale 但標示一般數值（非 10^n），範圍比資料 min/max 再放大一些
  - 沒有「真實曲線」可比較，配適優度用觀察值 vs. 配適值的 in-sample SMAPE / weighted RSS(log)

輸出：
  - 每組資料兩張圖：real_data_{name}_all_methods.png（六方法疊圖＋觀察值）
                     real_data_{name}_per_method.png（2×3 分面板）
  - ★ 新增：女生＋男生合併圖 Figure_esophageal_women_men_combined.png
    （格式完全比照模擬情境 Figure 3-6：A4 橫式、2x6、Women/Men 橫幅、無圖例、無 SMAPE）
  - 一份彙整表 real_data_summary.csv：2 組資料 × 6 方法的 in-sample SMAPE / weighted RSS(log)
  - 一份 Joinpoint 偵測到的轉折年份彙整（印在 console，也存進 summary 的附註）
================================================================
"""
import os
import hashlib
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
import numpy as np
import pandas as pd
import matplotlib; matplotlib.rcParams['axes.unicode_minus'] = False
import matplotlib.pyplot as plt
from matplotlib import ticker as mticker
import tensorflow as tf
tf.get_logger().setLevel('ERROR')

# ╔══════════════════════════════════════════════════╗
# ║              ★ 參數設定（比照模擬情境 / real_data_v1）★
# ╚══════════════════════════════════════════════════╝
EPOCHS   = int(os.environ.get('EPOCHS', 2000))
JP_PERM  = int(os.environ.get('JP_PERM', 199))
SEED = 42; NUM_NODES = 20; DL_SCALE = 2.0; LR = 0.005
N_AGE = 18

POP_CSV  = 'midyear1_pop.csv'
OUT_DIR  = os.environ.get('OUT_DIR', 'outputs')   # 相對路徑，Colab/本機都能用
os.makedirs(OUT_DIR, exist_ok=True)               # 資料夾不存在就自動建立

# ── 樣本外（walk-forward）驗證的設定 ──
#   N_ORIGINS  : 從資料尾端往回抓幾個「一步預測」起點（=幾個被預測的年份）
#   OOS_EPOCHS : 每個 fold 訓練 DL 模型的 epoch 數。可以跟主配適的 EPOCHS 不同，
#                預設略低是為了讓 2 組資料的樣本外驗證能在合理時間內跑完；
#                若要更嚴謹（跟主配適完全同設定），設 OOS_EPOCHS=$EPOCHS 即可。
N_ORIGINS  = int(os.environ.get('N_ORIGINS', 5))
OOS_EPOCHS = int(os.environ.get('OOS_EPOCHS', 1000))

device = '/gpu:0' if tf.config.list_physical_devices('GPU') else '/cpu:0'
print(f'EPOCHS={EPOCHS}  JP_PERM={JP_PERM}  device={device}')
print(f'樣本外驗證：N_ORIGINS={N_ORIGINS}  OOS_EPOCHS={OOS_EPOCHS}')

# ── 資料集：男女食道癌兩組 ──
DATASETS = [
    ('esophageal_male',   'esophageal1_male.csv',    1, 'Male esophageal cancer'),
    ('esophageal_female', 'esophageal2_female.csv', 2, 'Female esophageal cancer'),
]

# ★ 指定的 y 軸刻度範圍：男性 4–20，女性 0.4–2 ★
Y_RANGE_OVERRIDE = {
    'esophageal_male':   (4, 20),
    'esophageal_female': (0.4, 2),
}

# ══════════════════════════════════════════════════
# WHO 2000 世界標準人口權重（附錄四，2000 年欄）
# ══════════════════════════════════════════════════
WHO_W = np.array([8800, 8700, 8600, 8500, 8200, 7900, 7600, 7200,
                   6600, 6000, 5400, 4600, 3700, 3000, 2200, 1500, 900, 600], float)
WHO_W /= WHO_W.sum()

# ══════════════════════════════════════════════════
# 共用統計工具（與模擬情境 / real_data_v1 相同定義）
# ══════════════════════════════════════════════════
def _asir(d, n):
    return np.sum(WHO_W * d / n, axis=-1) * 100_000

def _vlog(d, n):
    ds = np.maximum(d, 0.5)
    a = np.sum(WHO_W * ds / n, axis=-1) * 100_000
    v = np.sum(WHO_W**2 * ds / n**2, axis=-1) * 100_000**2
    return v / a**2

def _smape(yt, yp):
    dd = (np.abs(yt) + np.abs(yp)) / 2
    return 100 * np.mean(np.abs(yp - yt) / (dd + 1e-9))

def _rcs_basis(x, k):
    K = len(k); c = [np.ones_like(x), x]
    for j in range(K - 2):
        tj, t2, t1 = k[j], k[K - 2], k[K - 1]; dd = t1 - tj
        c.append((np.maximum(0, x - tj) ** 3
                  - np.maximum(0, x - t2) ** 3 * (t1 - tj) / (t1 - t2)
                  + np.maximum(0, x - t1) ** 3 * (t2 - tj) / (t1 - t2)) / dd ** 2)
    return np.column_stack(c)

# ── Joinpoint（NCI 標準 ka–kb inward，與模擬情境完全相同）──
def _fit_piecewise(x, y, w, bp):
    bps = np.sort(bp)
    cols = [np.ones_like(x), x]
    for b in bps:
        cols.append(np.maximum(0, x - b))
    X = np.column_stack(cols)
    WX = X.T * w
    b_hat = np.linalg.solve(WX @ X + 1e-10 * np.eye(X.shape[1]), WX @ y)
    return b_hat, X @ b_hat

def _rss(x, y, w, bp):
    _, yhat = _fit_piecewise(x, y, w, bp)
    return np.sum(w * (y - yhat) ** 2)

def _best_jp_position(x, ly, inv, current_bps, candidates):
    best_rss = np.inf; best_bp = None
    for c in candidates:
        if any(abs(c - b) < 1e-9 for b in current_bps):
            continue
        r = _rss(x, ly, inv, current_bps + [c])
        if r < best_rss:
            best_rss = r; best_bp = c
    return best_bp, best_rss

def _permtest(x, ly, inv, bps_null, bps_alt, candidates, rng, n_perm):
    rss_null = _rss(x, ly, inv, bps_null)
    rss_alt  = _rss(x, ly, inv, bps_alt)
    T_obs = rss_null / rss_alt
    _, yhat_null = _fit_piecewise(x, ly, inv, bps_null)
    resid = ly - yhat_null
    count = 0
    for _ in range(n_perm):
        perm_ly = yhat_null + rng.permutation(resid)
        pr_null = _rss(x, perm_ly, inv, bps_null)
        bp_alt, _ = _best_jp_position(x, perm_ly, inv, bps_null, candidates)
        if bp_alt is None:
            pr_alt_rss = pr_null
        else:
            pr_alt_rss = _rss(x, perm_ly, inv, bps_null + [bp_alt])
        T_perm = pr_null / max(pr_alt_rss, 1e-12)
        if T_perm >= T_obs:
            count += 1
    return count / n_perm

def _joinpoint_select(x, ly, inv, max_jp=5, n_perm=JP_PERM, alpha=0.05, seed_extra=0):
    rng = np.random.default_rng(SEED + seed_extra)
    n = len(x)
    MAX = max_jp
    ka_end, kb_end = 3, 3
    candidates = x[ka_end: n - kb_end]
    if len(candidates) == 0 or max_jp == 0:
        return []
    bps_cache = {0: []}
    for k in range(1, max_jp + 1):
        prev = bps_cache[k - 1]
        new_bp, _ = _best_jp_position(x, ly, inv, prev, candidates)
        if new_bp is None:
            max_jp = k - 1
            break
        bps_cache[k] = sorted(prev + [new_bp])
    ka = 0
    kb = max_jp
    while ka < kb:
        p_val = _permtest(x, ly, inv, bps_cache[ka], bps_cache[kb], candidates, rng, n_perm)
        alpha_test = alpha / (MAX - ka)
        if p_val < alpha_test:
            ka += 1
        else:
            kb -= 1
    return bps_cache[ka]

def fit_joinpoint(x_obs, ly, inv, x_den, seed_extra=0):
    bps = _joinpoint_select(x_obs, ly, inv, seed_extra=seed_extra)
    b_hat, _ = _fit_piecewise(x_obs, ly, inv, bps)
    cols_d = [np.ones(len(x_den)), x_den]
    for b in np.sort(bps):
        cols_d.append(np.maximum(0, x_den - b))
    Xd = np.column_stack(cols_d)[:, :len(b_hat)]
    return np.exp(Xd @ b_hat), bps

# ── DL 模型（U-shaped RCS activation，與模擬情境完全相同架構）──
def _u_rcs(z):
    return 0.5 * (tf.pow(tf.nn.relu(z + 1.), 3) / 3.
                  - 2 * tf.pow(tf.nn.relu(z), 3) / 3.
                  + tf.pow(tf.nn.relu(z - 1.), 3) / 3. - z)

class DLModel(tf.keras.Model):
    def __init__(s, n, sv, seed, pb0, pbL, act='rcs', lc=1e-3, ct=1.0):
        super().__init__()
        bi = np.linspace(-sv * .95, sv * .95, NUM_NODES).astype('f')
        s.nb   = tf.Variable(np.tile(bi, (n, 1)))
        s.sc   = tf.Variable(tf.ones((n, 1, NUM_NODES)) * float(sv))
        s.beta = tf.Variable(tf.random.normal((n, NUM_NODES, 1), stddev=.1, seed=seed))
        s.bL   = tf.Variable(np.asarray(pbL, 'f').reshape(n, 1, 1))
        s.b0   = tf.Variable(np.asarray(pb0, 'f').reshape(n, 1, 1))
        s.av   = [s.nb, s.beta, s.bL, s.b0, s.sc]
        s.opt  = tf.keras.optimizers.Adam(LR)
        s.act = act; s.lc = lc; s.ct = ct
    def _fwd(s, x):
        xt = tf.tile(tf.expand_dims(x, 0), [s.nb.shape[0], 1, 1])
        z = xt * s.sc + tf.expand_dims(s.nb, 1)
        a = _u_rcs(z) if s.act == 'rcs' else tf.nn.relu(z)
        return tf.matmul(a, s.beta) + tf.matmul(xt, s.bL) + s.b0
    def _pen(s):
        r = tf.abs(s.nb) / (tf.abs(tf.squeeze(s.sc, 1)) + 1e-8)
        return tf.reduce_mean(tf.square(tf.nn.relu(r - s.ct)))
    @tf.function
    def train_step(s, x, ly, i2):
        with tf.GradientTape() as t:
            lp = s._fwd(x); L = tf.reduce_mean(tf.square(lp - ly) * i2)
            L += s.lc * s._pen()   # bowl penalty（唯一保留的正則項）
        s.opt.apply_gradients(zip(t.gradient(L, s.av), s.av))
        dc = tf.reduce_mean(tf.abs(s.sc)) * .95
        s.nb.assign(tf.clip_by_value(s.nb, -dc, dc))
        s.sc.assign(tf.clip_by_value(s.sc, .01, 15.))
    def predict(s, x): return np.exp(tf.squeeze(s._fwd(x), -1).numpy())

# ══════════════════════════════════════════════════
# y 軸：log scale、標示一般數值（非 10^n）、範圍放大
# ══════════════════════════════════════════════════
def _nice_yticks(ymin, ymax):
    # ★ 動態產生跨數量級的刻度候選（不再是寫死從 1 開始的清單）★
    #   涵蓋 0.01～數萬，每個級距用 1/1.2/1.4/1.6/1.8/2/2.5/3/3.5/4/4.5/5/6/7/8/9 分段，
    #   這樣不管資料是像男性食道癌落在 5～17，還是女性食道癌落在 0.6～1.2，
    #   都會有夠密的刻度可以讀出差異，不會出現「整段範圍沒有刻度」的狀況。
    base = [1, 1.2, 1.4, 1.6, 1.8, 2, 2.5, 3, 3.5, 4, 4.5, 5, 6, 7, 8, 9]
    cands = sorted(set(round(b * 10**e, 6) for e in range(-2, 4) for b in base))
    ticks = [v for v in cands if ymin <= v <= ymax]
    if len(ticks) > 9:   # 太密就稀疏化，避免擠成一團
        idx = sorted(set(np.linspace(0, len(ticks) - 1, 8).round().astype(int)))
        ticks = [ticks[i] for i in idx]
    return ticks

def _fmt_tick(v):
    if abs(v - round(v)) < 1e-9:
        return str(int(round(v)))
    s = f'{v:.2f}'.rstrip('0').rstrip('.')
    return s

def _apply_log_plain_yaxis(axx, ymin, ymax):
    axx.set_yscale('log')
    axx.set_ylim(ymin, ymax)
    yt = _nice_yticks(ymin, ymax)
    if len(yt) >= 2:
        axx.set_yticks(yt)
        axx.set_yticklabels([_fmt_tick(v) for v in yt])
    axx.yaxis.set_minor_formatter(mticker.NullFormatter())
    # 注意：不要再呼叫 set_major_formatter(ScalarFormatter())，
    # 那會把上面手動設定的整數字串標籤覆蓋掉，導致整數也被強制補成小數（例如 "4.0"）。

PUB = {
    'Linear'   : '#4A6FE3',
    'Quadratic': '#E0902A',
    'Trad-RCS' : '#2E9E5B',
    'Joinpoint': '#C0392B',
    'DL-ReLU'  : '#8E44AD',
    'DL-RCS'   : '#1B9E9E',
}
# ★ 全部線條統一調細、觀察點統一調小——線細了、點小了，
#   Joinpoint／DL-ReLU 本身固有的轉折棱角，跟 Trad-RCS／DL-RCS 的平滑，
#   自然就會在視覺上對比出來，不需要用顏色深淺或線寬去額外強調。
LW_LINE  = 1.3     # 跟模擬情境 Figure 3-6 的 prediction 線寬統一
OBS_S_OVERLAY = 10  # 疊圖：觀察點大小
OBS_S_PANEL   = 7   # 分面板圖：觀察點大小
PANEL_FIT_COLOR = '#E11010'  # 分面板圖：Fit 線顏色，跟模擬情境 Figure 3-6 的紅色統一
PREDS_ORDER = ['Linear', 'Quadratic', 'Trad-RCS', 'Joinpoint', 'DL-ReLU', 'DL-RCS']
RED = PANEL_FIT_COLOR
OBS_FACE = 'black'; OBS_EDGE = 'black'   # 觀察值：實心黑點

# 方法縮寫，跟模擬情境 Figure 3-6 的 Table 1 完全一致
METHOD_ABBR = {'Linear': 'LIN', 'Quadratic': 'QUAD', 'Trad-RCS': 'RCS',
               'Joinpoint': 'JP', 'DL-ReLU': 'NN-ReLU', 'DL-RCS': 'NN-RCS'}

# ══════════════════════════════════════════════════
# ★ 六方法「通用」配適函式：給定任意訓練子集 (x_tr, ly_tr, 權重)，
#   在任意預測點 x_pred 上配適＋預測。與 analyze_one() 裡主配適用的是
#   完全相同的公式／超參數，只是抽出來讓 walk-forward 驗證可以重複呼叫。
#   RCS 節點也是「只用訓練子集」重新算 percentile，模擬「當時只看得到這些資料」。
# ══════════════════════════════════════════════════
def _fit_all_methods(x_tr, ly_tr, inv_tr, invn_tr, x_pred, epochs, seed_extra=0):
    n_tr = len(x_tr)
    knots_tr = np.percentile(x_tr, np.linspace(5, 95, 5))

    Xr_tr = _rcs_basis(x_tr, knots_tr); Xr_pr = _rcs_basis(x_pred, knots_tr)
    Xl_tr = np.column_stack([np.ones(n_tr), x_tr])
    Xl_pr = np.column_stack([np.ones(len(x_pred)), x_pred])
    Xq_tr = np.column_stack([np.ones(n_tr), x_tr, x_tr**2])
    Xq_pr = np.column_stack([np.ones(len(x_pred)), x_pred, x_pred**2])

    wi = inv_tr
    XW = Xl_tr.T * wi; b_l = np.linalg.solve(XW @ Xl_tr, XW @ ly_tr)
    pb0, pbL = float(b_l[0]), float(b_l[1])
    out_l = np.exp(Xl_pr @ b_l)

    XW = Xq_tr.T * wi; b_q = np.linalg.solve(XW @ Xq_tr, XW @ ly_tr)
    out_q = np.exp(Xq_pr @ b_q)

    XW = Xr_tr.T * wi; b_r = np.linalg.solve(XW @ Xr_tr, XW @ ly_tr)
    out_r = np.exp(Xr_pr @ b_r)

    out_j, _ = fit_joinpoint(x_tr, ly_tr, wi, x_pred, seed_extra=seed_extra)

    xo_tf = tf.constant(x_tr.reshape(-1, 1), tf.float32)
    xp_tf = tf.constant(x_pred.reshape(-1, 1), tf.float32)
    lyt = tf.constant(ly_tr.reshape(1, -1, 1), tf.float32)
    i2t = tf.constant(invn_tr.reshape(1, -1, 1), tf.float32)

    sd = SEED + 300 + seed_extra
    tf.random.set_seed(sd); np.random.seed(sd)
    with tf.device(device):
        m = DLModel(1, DL_SCALE, sd, [pb0], [pbL], act='rcs')
        for _ in range(epochs):
            m.train_step(xo_tf, lyt, i2t)
    dl_rcs = m.predict(xp_tf).reshape(-1)

    sd2 = sd + 997
    tf.random.set_seed(sd2); np.random.seed(sd2)
    with tf.device(device):
        m2 = DLModel(1, DL_SCALE, sd2, [pb0], [pbL], act='relu')
        for _ in range(epochs):
            m2.train_step(xo_tf, lyt, i2t)
    dl_relu = m2.predict(xp_tf).reshape(-1)

    return {'Linear': out_l, 'Quadratic': out_q, 'Trad-RCS': out_r,
            'Joinpoint': out_j, 'DL-ReLU': dl_relu, 'DL-RCS': dl_rcs}

# ══════════════════════════════════════════════════
# ★ 樣本外驗證：expanding-window、一步預測（walk-forward）★
# ══════════════════════════════════════════════════
def walk_forward_smape(x_obs, ly, inv, invn, asir_obs, years,
                        n_origins=N_ORIGINS, epochs=OOS_EPOCHS, seed_extra=0):
    T = len(x_obs)
    n_origins = max(1, min(n_origins, T - 10))  # 至少保留 10 年做訓練起點下限保護
    per_method_smape = {mn: [] for mn in PREDS_ORDER}
    detail_rows = []
    for h in range(n_origins, 0, -1):
        cut = T - h                      # 只用前 cut 年配適
        x_tr, ly_tr = x_obs[:cut], ly[:cut]
        inv_tr, invn_tr = inv[:cut], invn[:cut]
        x_te = x_obs[cut:cut + 1]        # 預測「下一年」（單步外插）
        y_te_true = asir_obs[cut]
        test_year = int(years[cut])

        preds = _fit_all_methods(x_tr, ly_tr, inv_tr, invn_tr, x_te, epochs,
                                  seed_extra=seed_extra * 100 + h)
        for mn in PREDS_ORDER:
            yp = float(preds[mn][0])
            sm = _smape(np.array([y_te_true]), np.array([yp]))
            per_method_smape[mn].append(sm)
            detail_rows.append(dict(origin_year=test_year, n_train_years=cut,
                                     method=mn, observed=y_te_true,
                                     predicted=yp, smape=sm))
    summary = {mn: float(np.mean(per_method_smape[mn])) for mn in PREDS_ORDER}
    return summary, pd.DataFrame(detail_rows)

matplotlib.rcParams.update({
    'font.family': 'serif',
    # ★ 鎖定單一字型（不再是 Times New Roman / DejaVu Serif 的 fallback 清單）★
    #   之前用 fallback 清單時，如果兩次執行的機器上有沒有裝 Times New Roman
    #   的狀況不同，字體粗細看起來就會不一樣（這正是模擬情境 Figure 3-6
    #   跟真實資料圖字體粗細對不起來的原因）。改成只用 DejaVu Serif，
    #   不管在哪台機器、哪次執行，畫出來的字一定是同一個粗細。
    'font.serif': ['DejaVu Serif'],
    'font.size': 10,
    'axes.grid': False,
    'axes.spines.top': False,
    'axes.spines.right': False,
    # ★ 邊框／刻度線寬統一設成全域預設，跟模擬情境 Figure 3-6 一致 ★
    #   這樣所有圖（疊圖、per_method、合併圖）都不用再各自手動設定，
    #   直接繼承這個全域值，不會有些圖粗、有些圖細的落差。
    'axes.linewidth': 0.9,
    'xtick.major.width': 0.9,
    'ytick.major.width': 0.9,
})

# ══════════════════════════════════════════════════
# 讀取人口母檔（一次即可，之後依性別篩選）
# ══════════════════════════════════════════════════
def load_population():
    pop = pd.read_csv(POP_CSV, encoding='big5')
    tpop_cols = [f'TPOP{i}' for i in range(1, N_AGE + 1)]
    pop = pop[['year', 'SEX'] + tpop_cols].copy()
    pop['year'] = pop['year'].astype(int)
    return pop, tpop_cols

def load_cancer_counts(csv_path):
    raw = pd.read_excel(csv_path, header=None)
    years_row = raw.iloc[4, 1:46].values.astype(int)
    # ★ 動態抓 age_level 資料列，不假設固定 18 列 ★
    age_levels, data_rows = [], []
    for i in range(5, raw.shape[0]):
        val = raw.iloc[i, 0]
        if pd.isna(val):
            continue
        try:
            al = int(val)
        except (ValueError, TypeError):
            continue  # 跳過「總計」列等非數字列
        if not (1 <= al <= N_AGE):
            continue
        age_levels.append(al)
        data_rows.append(raw.iloc[i, 1:46].values.astype(float))

    full = np.zeros((N_AGE, len(years_row)))
    for row, al in zip(data_rows, age_levels):
        full[al - 1, :] = row

    missing = sorted(set(range(1, N_AGE + 1)) - set(age_levels))
    if missing:
        print(f'  ⚠ {csv_path}：age_level {missing} 全年份 0 例（SAS FREQ 表格中被省略），已補 0')

    df = pd.DataFrame(full.T, columns=[f'age{i}' for i in range(1, N_AGE + 1)])
    df.insert(0, 'year', years_row)
    return df

# ══════════════════════════════════════════════════
# 單一資料集：合併 → 配適六方法 → 存圖 → 回傳彙整列
# ══════════════════════════════════════════════════
def analyze_one(name, csv_path, sex_code, label, pop_all, tpop_cols):
    print(f'\n{"="*70}\n{name}  ({label})\n{"="*70}')

    pop_s = pop_all[pop_all['SEX'] == sex_code][['year'] + tpop_cols] \
              .sort_values('year').reset_index(drop=True)
    cancer = load_cancer_counts(csv_path)

    common_years = sorted(set(pop_s['year']).intersection(set(cancer['year'])))
    pop_s = pop_s[pop_s['year'].isin(common_years)].sort_values('year').reset_index(drop=True)
    cancer = cancer[cancer['year'].isin(common_years)].sort_values('year').reset_index(drop=True)
    assert list(pop_s['year']) == list(cancer['year'])

    years = np.array(common_years, dtype=int)
    T = len(years)
    YEAR_START, YEAR_END = int(years.min()), int(years.max())
    n_mat = pop_s[tpop_cols].values.astype(float)
    d_mat = cancer[[f'age{i}' for i in range(1, N_AGE + 1)]].values.astype(float)
    print(f'共同年份：{YEAR_START}–{YEAR_END}（T={T} 年）  總個案數={int(d_mat.sum())}')

    asir_obs = _asir(d_mat, n_mat)
    s2_log   = _vlog(d_mat, n_mat)
    inv      = 1 / (s2_log + 1e-12)
    invn     = inv / (inv.mean() + 1e-12)
    ly       = np.log(np.maximum(asir_obs, 1e-3))

    x_obs = np.linspace(-1, 1, T)
    x_den = np.linspace(-1, 1, 300); N_DEN = len(x_den)
    years_den = np.linspace(YEAR_START, YEAR_END, N_DEN)
    knots = np.percentile(x_obs, np.linspace(5, 95, 5))

    Xr_o = _rcs_basis(x_obs, knots); Xr_d = _rcs_basis(x_den, knots)
    Xl_o = np.column_stack([np.ones(T), x_obs])
    Xl_d = np.column_stack([np.ones(N_DEN), x_den])
    Xq_o = np.column_stack([np.ones(T), x_obs, x_obs**2])
    Xq_d = np.column_stack([np.ones(N_DEN), x_den, x_den**2])

    xo_tf = tf.constant(x_obs.reshape(-1, 1), tf.float32)
    xd_tf = tf.constant(x_den.reshape(-1, 1), tf.float32)

    # -- Linear / Quadratic / Trad-RCS：WLS 閉式解 --
    wi = inv
    XW = Xl_o.T * wi; b_l = np.linalg.solve(XW @ Xl_o, XW @ ly)
    pb0, pbL = float(b_l[0]), float(b_l[1])
    out_l = np.exp(Xl_d @ b_l)

    XW = Xq_o.T * wi; b_q = np.linalg.solve(XW @ Xq_o, XW @ ly)
    out_q = np.exp(Xq_d @ b_q)

    XW = Xr_o.T * wi; b_r = np.linalg.solve(XW @ Xr_o, XW @ ly)
    out_r = np.exp(Xr_d @ b_r)

    # -- Joinpoint --
    seed_extra = int(hashlib.md5(name.encode()).hexdigest(), 16) % 10_000
    out_j, jp_bps = fit_joinpoint(x_obs, ly, wi, x_den, seed_extra=seed_extra)
    jp_years = [round(YEAR_START + (b + 1) / 2 * (YEAR_END - YEAR_START)) for b in jp_bps]
    print(f'  Joinpoint 轉折年份：{jp_years}')

    # -- DL-RCS / DL-ReLU（N=1）--
    lyt = tf.constant(ly.reshape(1, -1, 1), tf.float32)
    i2t = tf.constant(invn.reshape(1, -1, 1), tf.float32)

    sd = SEED + 200 + seed_extra
    tf.random.set_seed(sd); np.random.seed(sd)
    with tf.device(device):
        m = DLModel(1, DL_SCALE, sd, [pb0], [pbL], act='rcs')
        for _ in range(EPOCHS):
            m.train_step(xo_tf, lyt, i2t)
    dl_rcs = m.predict(xd_tf).reshape(-1)

    sd2 = sd + 997
    tf.random.set_seed(sd2); np.random.seed(sd2)
    with tf.device(device):
        m2 = DLModel(1, DL_SCALE, sd2, [pb0], [pbL], act='relu')
        for _ in range(EPOCHS):
            m2.train_step(xo_tf, lyt, i2t)
    dl_relu = m2.predict(xd_tf).reshape(-1)

    CURVES_DEN = {'Linear': out_l, 'Quadratic': out_q, 'Trad-RCS': out_r,
                  'Joinpoint': out_j, 'DL-ReLU': dl_relu, 'DL-RCS': dl_rcs}
    fit_at_obs = {mn: np.interp(x_obs, x_den, CURVES_DEN[mn]) for mn in PREDS_ORDER}

    rows = []
    print(f'{"方法":<10}{"Weighted RSS(log)":>20}{"SMAPE":>12}')
    for mn in PREDS_ORDER:
        wrss = np.sum(inv * (ly - np.log(np.maximum(fit_at_obs[mn], 1e-3))) ** 2)
        sm = _smape(asir_obs, fit_at_obs[mn])
        print(f'{mn:<10}{wrss:>20.4f}{sm:>11.2f}%')
        rows.append(dict(dataset=name, label=label, method=mn,
                          weighted_rss_log=wrss, smape=sm,
                          year_start=YEAR_START, year_end=YEAR_END, n_years=T,
                          joinpoint_years=';'.join(map(str, jp_years)) if mn == 'Joinpoint' else ''))

    # -- ★ 樣本外驗證（walk-forward，一步預測）★ --
    print(f'\n  [樣本外驗證] walk-forward，N_ORIGINS={N_ORIGINS}（預測最後 {N_ORIGINS} 個未看過的年份）')
    oos_summary, oos_detail = walk_forward_smape(x_obs, ly, inv, invn, asir_obs, years,
                                                  seed_extra=seed_extra)
    oos_detail.insert(0, 'dataset', name); oos_detail.insert(1, 'label', label)
    print(f'  {"方法":<10}{"OOS SMAPE (walk-forward)":>28}{"In-sample SMAPE":>20}')
    oos_rows = []
    for mn in PREDS_ORDER:
        insample_sm = _smape(asir_obs, fit_at_obs[mn])
        print(f'  {mn:<10}{oos_summary[mn]:>27.2f}%{insample_sm:>19.2f}%')
        oos_rows.append(dict(dataset=name, label=label, method=mn,
                              oos_smape_walk_forward=oos_summary[mn],
                              insample_smape=insample_sm,
                              n_origins=N_ORIGINS))

    # ── 圖一：疊圖 ──
    ALLV = np.concatenate([asir_obs] + [CURVES_DEN[mn] for mn in PREDS_ORDER])
    ALLV = ALLV[ALLV > 0]
    if name in Y_RANGE_OVERRIDE:
        YMIN, YMAX = Y_RANGE_OVERRIDE[name]   # 指定的固定範圍
    else:
        YMIN, YMAX = ALLV.min() * 0.55, ALLV.max() * 1.6   # 沒指定就自動放寬

    fig1, ax = plt.subplots(figsize=(8, 5.2))
    ax.scatter(years, asir_obs, s=OBS_S_OVERLAY, facecolor=OBS_FACE, edgecolor=OBS_EDGE,
               linewidth=0.8, zorder=6, label='Observed ASIR')
    for mn in PREDS_ORDER:
        ax.plot(years_den, CURVES_DEN[mn], color=PUB[mn], lw=LW_LINE,
                 solid_capstyle='round', label=mn, zorder=4)
    _apply_log_plain_yaxis(ax, YMIN, YMAX)
    ax.set_xlabel('Year'); ax.set_ylabel('ASIR (/100,000)')
    ax.set_title(f'{label}, age-standardized incidence rate\n'
                 f'({YEAR_START}\u2013{YEAR_END}, WHO 2000 standard population)',
                 fontsize=10.5, fontweight='bold')
    ax.legend(fontsize=7.5, loc='best', frameon=True, framealpha=0.9, ncol=2)
    plt.tight_layout()
    fig1.savefig(f'{OUT_DIR}/real_data_{name}_all_methods.png', dpi=200,
                 bbox_inches='tight', facecolor='white')
    plt.close(fig1)

    # ── 圖二：2×3 分面板（統一同一個顏色、統一細線寬，靠線細+點小讓轉折自然顯現）──
    fig2, axes = plt.subplots(2, 3, figsize=(11.69, 6.5), squeeze=False)
    for idx, mn in enumerate(PREDS_ORDER):
        r, c = divmod(idx, 3)
        axx = axes[r][c]
        axx.scatter(years, asir_obs, s=OBS_S_PANEL, facecolor=OBS_FACE, edgecolor=OBS_EDGE,
                    linewidth=0.7, zorder=3, label='Observed' if idx == 0 else None)
        axx.plot(years_den, CURVES_DEN[mn], color=PANEL_FIT_COLOR, lw=LW_LINE,
                  solid_capstyle='round', zorder=4, label='Fit' if idx == 0 else None)
        _apply_log_plain_yaxis(axx, YMIN, YMAX)
        if c != 0:
            axx.set_yticklabels([])
        axx.set_xlim(YEAR_START - .5, YEAR_END + .5)
        axx.set_xticks([YEAR_START, (YEAR_START + YEAR_END) // 2, YEAR_END])
        sm = _smape(asir_obs, fit_at_obs[mn])
        axx.set_title(f'{mn}  (in-sample SMAPE {sm:.2f}%)', fontsize=9, fontweight='bold')
        if c == 0:
            axx.set_ylabel('ASIR (/100,000)')
        if r == 1:
            axx.set_xlabel('Year')
        if idx == 0:
            axx.legend(fontsize=7, loc='upper left', frameon=True, framealpha=0.9)
    fig2.suptitle(f'{label} real data — six-method fit ({YEAR_START}\u2013{YEAR_END})',
                  fontsize=11, fontweight='bold', y=1.02)
    plt.tight_layout()
    fig2.savefig(f'{OUT_DIR}/real_data_{name}_per_method.png', dpi=200,
                 bbox_inches='tight', facecolor='white')
    plt.close(fig2)

    # ★ 新增：把畫「Women/Men 合併圖」需要的資料回傳出去 ★
    plot_data = dict(
        label=label, CURVES_DEN=CURVES_DEN, years_den=years_den,
        asir_obs=asir_obs, years=years, YMIN=YMIN, YMAX=YMAX,
        YEAR_START=YEAR_START, YEAR_END=YEAR_END,
    )
    return rows, oos_rows, oos_detail, plot_data


# ══════════════════════════════════════════════════
# ★ 新增：女生＋男生合併圖，格式完全比照模擬情境 Figure 3-6 ★
# ══════════════════════════════════════════════════
def plot_women_men_combined(data_female, data_male, out_path):
    """
    2 排 x 6 欄：上排 Women（女性食道癌），下排 Men（男性食道癌）。
    格式與線條/字體風格完全比照模擬情境 Figure 3-6：
      - A4 橫式 figsize=(11.69, 8.27)
      - 邊框線寬 0.7（跟 Figure 3-6 的 spine linewidth 一致）
      - 欄標題（LIN/QUAD/RCS/JP/NN-ReLU/NN-RCS）只在最上面出現一次
      - 左上角橫寫 "Women"，兩排中間橫寫 "Men"
      - 縱軸標籤 spelled-out，兩排各自維持教授指定的 y 軸範圍
      - 橫軸只在最下面（Men 那排）顯示；改成固定 4 個刻度
        （1980 / 1991 / 2002 / 2023，跟其他真實資料圖一致），
        並加大欄與欄之間的間距，避免相鄰 panel 的年份數字互相重疊
      - 無圖例、無 SMAPE
    """
    fig, axes = plt.subplots(2, 6, figsize=(11.69, 8.27), squeeze=False)
    ROW_DATA = [data_female, data_male]   # row0=Women, row1=Men
    ROW_LABELS = ['Women', 'Men']

    for row, d in enumerate(ROW_DATA):
        for col, mn in enumerate(PREDS_ORDER):
            axx = axes[row][col]
            axx.scatter(d['years'], d['asir_obs'], s=OBS_S_PANEL,
                        facecolor=OBS_FACE, edgecolor=OBS_EDGE,
                        linewidth=0.7, zorder=3)
            axx.plot(d['years_den'], d['CURVES_DEN'][mn], color=PANEL_FIT_COLOR,
                     lw=LW_LINE, solid_capstyle='round', zorder=4)
            _apply_log_plain_yaxis(axx, d['YMIN'], d['YMAX'])
            if col != 0:
                axx.set_yticklabels([])
            axx.set_xlim(d['YEAR_START'] - .5, d['YEAR_END'] + .5)

            # fixed 4-tick scheme, consistent with the other real-data figures,
            # instead of np.linspace's 5 ticks (which were crowding into each other)
            yr0, yr1 = d['YEAR_START'], d['YEAR_END']
            xticks = [yr0, yr0 + round((yr1 - yr0) / 3),
                      yr0 + round(2 * (yr1 - yr0) / 3), yr1]
            axx.set_xticks(xticks)

            if row == 1:
                axx.set_xticklabels([str(v) for v in xticks], fontsize=8)
                axx.set_xlabel('Calendar Year', fontsize=9)
            else:
                axx.set_xticklabels([])
            if row == 0:
                axx.set_title(METHOD_ABBR[mn], fontsize=11, fontweight='bold', pad=8)
            if col == 0:
                axx.set_ylabel('Age-Standardized Incidence Rate\n(per 100,000)', fontsize=8.5)

    # wider gaps between columns so the 4-digit year labels of neighboring
    # panels no longer run into each other
    fig.subplots_adjust(left=0.07, right=0.99, bottom=0.08, top=0.90,
                         wspace=0.30, hspace=0.35)

    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    title_tops = [axes[0, c].title.get_window_extent(renderer=renderer)
                  .transformed(fig.transFigure.inverted()).y1 for c in range(6)]
    row0_title_top = max(title_tops)
    fig.text(0.02, row0_title_top + 0.025, ROW_LABELS[0],
              ha='left', va='bottom', fontsize=13, fontweight='bold')

    row0_bottom = axes[0, 0].get_position().y0
    row1_top = axes[1, 0].get_position().y1
    mid_y = (row0_bottom + row1_top) / 2
    fig.text(0.02, mid_y, ROW_LABELS[1], ha='left', va='center',
              fontsize=13, fontweight='bold')

    fig.savefig(out_path, dpi=200, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f'Saved combined Women/Men figure: {out_path}')


# ══════════════════════════════════════════════════
# 主程式：跑全部資料
# ══════════════════════════════════════════════════
if __name__ == '__main__':
    pop_all, tpop_cols = load_population()
    all_rows, all_oos_rows, all_oos_detail = [], [], []
    plot_data_by_name = {}
    for name, csv_path, sex_code, label in DATASETS:
        rows, oos_rows, oos_detail, plot_data = analyze_one(
            name, csv_path, sex_code, label, pop_all, tpop_cols)
        all_rows.extend(rows)
        all_oos_rows.extend(oos_rows)
        all_oos_detail.append(oos_detail)
        plot_data_by_name[name] = plot_data

    # ★ 新增：女生＋男生合併圖（格式比照 Figure 3-6）★
    if 'esophageal_female' in plot_data_by_name and 'esophageal_male' in plot_data_by_name:
        plot_women_men_combined(
            plot_data_by_name['esophageal_female'],
            plot_data_by_name['esophageal_male'],
            f'{OUT_DIR}/Figure_esophageal_women_men_combined.png'
        )

    summary = pd.DataFrame(all_rows)
    summary.to_csv(f'{OUT_DIR}/real_data_summary.csv', index=False, encoding='utf-8-sig')

    oos_summary_df = pd.DataFrame(all_oos_rows)
    oos_summary_df.to_csv(f'{OUT_DIR}/real_data_oos_summary.csv', index=False, encoding='utf-8-sig')
    oos_detail_df = pd.concat(all_oos_detail, ignore_index=True)
    oos_detail_df.to_csv(f'{OUT_DIR}/real_data_oos_detail.csv', index=False, encoding='utf-8-sig')

    # ── 總彙整表（console）──
    print(f'\n{"="*90}')
    print('In-sample SMAPE 彙整（%）— 資料 × 6 方法')
    piv = summary.pivot(index='dataset', columns='method', values='smape')[PREDS_ORDER]
    print(piv.round(2).to_string())

    print(f'\n{"="*90}')
    print(f'樣本外 walk-forward SMAPE 彙整（%）— 資料 × 6 方法（N_ORIGINS={N_ORIGINS}）')
    piv_oos = oos_summary_df.pivot(index='dataset', columns='method',
                                    values='oos_smape_walk_forward')[PREDS_ORDER]
    print(piv_oos.round(2).to_string())
    print(f'{"="*90}')

    # ── 比較圖：In-sample vs. 樣本外（walk-forward），跨資料平均，每個方法一根長條 ──
    avg_insample = oos_summary_df.groupby('method')['insample_smape'].mean().reindex(PREDS_ORDER)
    avg_oos = oos_summary_df.groupby('method')['oos_smape_walk_forward'].mean().reindex(PREDS_ORDER)

    fig3, ax3 = plt.subplots(figsize=(8, 5))
    xpos = np.arange(len(PREDS_ORDER)); w = 0.35
    ax3.bar(xpos - w/2, avg_insample.values, width=w, color='#888888', label='In-sample SMAPE')
    ax3.bar(xpos + w/2, avg_oos.values, width=w, color=RED, label='OOS walk-forward SMAPE')
    ax3.set_xticks(xpos); ax3.set_xticklabels(PREDS_ORDER, fontsize=9)
    ax3.set_ylabel('SMAPE (%)')
    ax3.set_title(f'In-sample vs. out-of-sample (walk-forward) SMAPE\n'
                   f'averaged across real-data series (N_ORIGINS={N_ORIGINS})',
                   fontsize=10.5, fontweight='bold')
    ax3.legend(fontsize=8.5, frameon=True, framealpha=0.9)
    plt.tight_layout()
    fig3.savefig(f'{OUT_DIR}/real_data_oos_vs_insample.png', dpi=200,
                 bbox_inches='tight', facecolor='white')
    plt.close(fig3)

    print(f'\n完成，輸出於 {OUT_DIR}/：')
    for name, *_ in DATASETS:
        print(f'  real_data_{name}_all_methods.png')
        print(f'  real_data_{name}_per_method.png')
    print('  Figure_esophageal_women_men_combined.png （★新增：女＋男合併圖，比照 Figure 3-6）')
    print('  real_data_summary.csv                 （in-sample SMAPE / weighted RSS）')
    print('  real_data_oos_summary.csv              （資料 × 6 方法，OOS vs in-sample SMAPE）')
    print('  real_data_oos_detail.csv               （每個 walk-forward 起點、每個方法的逐筆預測明細）')
    print('  real_data_oos_vs_insample.png          （跨資料平均，in-sample vs OOS 長條圖）')

EPOCHS=2000  JP_PERM=199  device=/cpu:0
樣本外驗證：N_ORIGINS=5  OOS_EPOCHS=1000

esophageal_male  (Male esophageal cancer)
共同年份：1980–2023（T=44 年）  總個案數=60852
  Joinpoint 轉折年份：[1988, 2008, 2013]
方法           Weighted RSS(log)       SMAPE
Linear                812.8643      11.05%
Quadratic             567.5429      10.72%
Trad-RCS               56.9869       3.19%
Joinpoint              66.6744       3.75%
DL-ReLU                49.1437       3.01%
DL-RCS                 54.1192       3.14%

  [樣本外驗證] walk-forward，N_ORIGINS=5（預測最後 5 個未看過的年份）
  方法            OOS SMAPE (walk-forward)     In-sample SMAPE
  Linear                          19.56%              11.05%
  Quadratic                       14.23%              10.72%
  Trad-RCS                         3.04%               3.19%
  Joinpoint                        2.26%               3.75%
  DL-ReLU                          1.74%               3.01%
  DL-RCS                           3.09%               3.14%

esophageal_female  (Female eso